In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

In [ ]:
# 2. Батч-обработка STL из Компас (папка с файлами)
INPUT_DIR = r"C:\path\to\your_stl_files"  # поменять путь на нормальный
output_path = Path("dataset")
output_path.mkdir(exist_ok=True)

stl_files = glob.glob(os.path.join(INPUT_DIR, "*.stl"))
data = []

for stl in stl_files[:50]:  # 50 файлов для теста
    mesh = trimesh.load(stl)
    data.append({
        'file': Path(stl).name,
        'volume': mesh.volume,
        'area': mesh.area,
        'faces': len(mesh.faces),
        'points': len(mesh.vertices),  # Для ML
    })
    print(f"✓ {Path(stl).name}: vol={mesh.volume:.2f}")

df = pd.DataFrame(data)
df.to_csv(output_path / "cad_data.csv", index=False)
print(df.head())
df.describe()

In [ ]:
# 3. Визуализация 1-го файла
mesh = trimesh.load(stl_files[0])
fig = plt.figure(figsize=(12,4))
ax1 = fig.add_subplot(121); plt.plot(df['volume'], df['area'], 'o'); plt.xlabel('Volume'); plt.ylabel('Area')
ax2 = fig.add_subplot(122, projection='3d'); ax2.scatter(mesh.vertices[:,0], mesh.vertices[:,1], mesh.vertices[:,2], s=1)
plt.show()

In [ ]:
# 4. Сохрани points для ML (npz датасет)
for i, row in df.iterrows():
    mesh = trimesh.load(os.path.join(INPUT_DIR, row['file']))
    points = mesh.sample(1024)  # 1024 точки
    np.savez(f"dataset/points_{i}.npz", points=points, volume=row['volume'])
print("Датасет готов для torch/sklearn!")